# Phase 6 — Uncertainty Simulation and Scenario Stress Testing

## Business purpose

The earlier phases of this project established a strong next-day forecasting benchmark, a decision-support layer, multi-step planning horizons, and a dedicated breach-detection classifier. However, operational leaders usually need more than a single predicted value. They also need to understand uncertainty and how forecast outcomes may change under stressed operating conditions.

This notebook extends the project from point forecasting into uncertainty-aware planning. Instead of asking only, "What is the predicted dwell?", Phase 6 asks, "What range of dwell outcomes is plausible, and how likely is a 24-hour breach under normal and stressed conditions?"

## What this notebook does

This notebook keeps the tuned LightGBM regression model as the core forecasting engine and adds a simulation layer on top of it. The simulation repeatedly perturbs key operational inputs within reasonable bounds and produces a distribution of predicted dwell outcomes.

The notebook then uses those simulated outcomes to estimate:

- central tendency of predicted dwell
- uncertainty bands
- breach probability above 24 hours
- response under stress scenarios such as increased inbound load or reduced crew availability

## Why it matters

This phase moves the project closer to practical planning support. It helps translate a point forecast into a more decision-relevant view of risk and uncertainty, which is more aligned with how real operations planning is performed.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", None)
np.random.seed(42)

figures_dir = Path("../reports/figures")
figures_dir.mkdir(parents=True, exist_ok=True)

## 1. Load data

We begin by loading the synthetic terminal dwell dataset used throughout the project. The dataset is sorted by `terminal_id` then `date` to align with the convention used in earlier project notebooks.

In [ ]:
df = pd.read_csv("../data/synthetic/phase1_terminal_dwell.csv", parse_dates=["date"])
df = df.sort_values(["terminal_id", "date"]).reset_index(drop=True)

print("Dataset shape:", df.shape)
df.head()

## 2. Prepare the official Phase 1 feature set

To stay consistent with the main forecasting benchmark, this notebook uses the same official Phase 1 target, features, and split date.

Important modeling choices remain unchanged:

- `target_dwell_hours` is the prediction target
- `region` is not included in the official feature set
- `terminal_id` is treated as a native categorical feature for LightGBM
- the split date remains `2024-07-01`

This preserves comparability with the earlier tuned LightGBM benchmark.

In [3]:
target_col = "target_dwell_hours"

feature_cols = [
    "terminal_id",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct",
    "is_weekend",
    "month",
]

split_date = "2024-07-01"

train_df = df[df["date"] < split_date].copy()
test_df = df[df["date"] >= split_date].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Train shape: (7296, 13)
Test shape: (1464, 13)


In [4]:
train_df["terminal_id"] = train_df["terminal_id"].astype("category")
test_df["terminal_id"] = test_df["terminal_id"].astype("category")

X_train = train_df[feature_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

## 3. Train the tuned LightGBM benchmark model

Before introducing simulation, this notebook rebuilds the tuned LightGBM regression benchmark that emerged as the strongest next-day dwell forecasting model in Phase 1.

The goal here is not to change the benchmark, but to reuse it as the forecasting engine underneath the simulation layer.

In [6]:
reg_model = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=200,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=20,
    random_state=42
)

reg_model.fit(
    X_train,
    y_train,
    categorical_feature=["terminal_id"]
)

test_pred = reg_model.predict(X_test)

mse = mean_squared_error(y_test, test_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, test_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE: {mae:.3f}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000114 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1066
[LightGBM] [Info] Number of data points in the train set: 7296, number of used features: 9
[LightGBM] [Info] Start training from score 19.060458
Test RMSE: 3.749
Test MAE: 2.879


## 4. Select sample cases for simulation

To keep the first demonstration focused and interpretable, the notebook begins with a small random sample of three rows drawn from the test set.

These rows are illustrative only. They are not intended to be representative of the full holdout population. Their purpose is to show how the simulation function produces uncertainty bands and breach probability estimates for individual terminal-days.

In [7]:
sample_cases = test_df.sample(3, random_state=42).copy()

sample_cases[[
    "date",
    "terminal_id",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct",
    "target_dwell_hours"
]]

,date,terminal_id,inbound_train_count,inbound_car_count,cars_on_hand,yard_occupancy_pct,crew_starts_available,locomotive_availability_pct,target_dwell_hours
8592,2024-12-10,T01,5,114,90,10.0,14,86.2,15.1
7471,2024-07-22,T08,7,141,52,13.0,11,92.2,16.8
7571,2024-08-04,T04,16,341,931,98.0,17,86.4,24.1


The three rows above span a range of observed dwell outcomes and include terminals from different parts of the test period. They serve as convenient demonstrations, not as a curated or representative sample.

## 5. Build a baseline Monte Carlo simulation function

The simulation function perturbs selected operational inputs around their observed values using bounded random variation. Each perturbed version of the row is scored by the tuned regression model, producing a distribution of predicted dwell outcomes.

This is a simple first-step simulation layer, not a full operational simulator. Its role is to estimate uncertainty around the modeled operating state and provide a practical breach-probability view.

### Perturbation noise assumptions

The perturbation scales used in this simulation are first-pass heuristic assumptions designed to represent plausible day-to-day operational variability. They have not been formally estimated from historical data.

Each perturbed feature is scaled by a multiplicative normal factor `Normal(1.0, σ)`. Noise is set higher for volume and count features (σ = 0.08–0.10), where day-to-day variability is relatively larger, and lower for percentage-bounded features (σ = 0.04–0.05), which tend to vary less in relative terms over a single planning horizon. All results are clipped or floored at zero to enforce physical bounds.

In a production context, these scales should be replaced with empirically derived estimates based on observed day-to-day variance in each operational measure.

In [ ]:
def simulate_dwell_distribution(
    model,
    row,
    feature_cols,
    n_simulations=1000,
    breach_threshold=24.0
):
    base = row[feature_cols]
    n = n_simulations

    sim_df = pd.DataFrame({
        "terminal_id":                 [base["terminal_id"]] * n,
        "inbound_train_count":         np.maximum(0, np.round(
                                           base["inbound_train_count"] * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "inbound_car_count":           np.maximum(0, np.round(
                                           base["inbound_car_count"] * np.random.normal(1.00, 0.10, n)
                                       )).astype(int),
        "cars_on_hand":                np.maximum(0, np.round(
                                           base["cars_on_hand"] * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "yard_occupancy_pct":          np.clip(
                                           base["yard_occupancy_pct"] * np.random.normal(1.00, 0.05, n),
                                           0, 100
                                       ),
        "crew_starts_available":       np.maximum(0, np.round(
                                           base["crew_starts_available"] * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "locomotive_availability_pct": np.clip(
                                           base["locomotive_availability_pct"] * np.random.normal(1.00, 0.04, n),
                                           0, 100
                                       ),
        "is_weekend":                  np.full(n, int(base["is_weekend"])),
        "month":                       np.full(n, int(base["month"])),
    })[feature_cols]

    sim_df["terminal_id"] = sim_df["terminal_id"].astype("category")

    simulated_preds = model.predict(sim_df)

    summary = {
        "mean_pred":          float(simulated_preds.mean()),
        "median_pred":        float(np.median(simulated_preds)),
        "p10":                float(np.percentile(simulated_preds, 10)),
        "p90":                float(np.percentile(simulated_preds, 90)),
        "min_pred":           float(simulated_preds.min()),
        "max_pred":           float(simulated_preds.max()),
        "breach_probability": float((simulated_preds >= breach_threshold).mean()),
    }

    return simulated_preds, summary

## 6. Run one severe-breach example

This first simulation case uses the row with the highest observed dwell in the test set. Starting with a severe realized breach makes the regression model's compression visible and establishes an important interpretation about where simulation is and is not reliable.

The output includes both the full simulated prediction distribution and a compact summary of key planning statistics.

In [16]:
high_risk_cases = test_df[test_df["target_dwell_hours"] >= 22].copy()

print("Number of high-risk candidate rows:", len(high_risk_cases))

example_row = high_risk_cases.sort_values("target_dwell_hours", ascending=False).iloc[0].copy()

example_row[[
    "date",
    "terminal_id",
    "target_dwell_hours",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct"
]]

simulated_preds, sim_summary = simulate_dwell_distribution(
    model=reg_model,
    row=example_row,
    feature_cols=feature_cols,
    n_simulations=1000,
    breach_threshold=24.0
)

sim_summary



Number of high-risk candidate rows: 380


{'mean_pred': np.float64(20.815415313457642),
 'median_pred': np.float64(20.8422476443979),
 'p10': np.float64(20.1325320550737),
 'p90': np.float64(21.449108897784193),
 'min_pred': np.float64(19.478236247285366),
 'max_pred': np.float64(21.90383076980617),
 'breach_probability': np.float64(0.0)}

## 7. Show the severe-breach simulated dwell distribution

The histogram plots the distribution of simulated predicted dwell for the severe-breach row. The dashed vertical line marks the 24-hour threshold and the solid line marks the mean simulated prediction.

When the entire distribution sits well below the threshold, it signals that the regression model is compressing an extreme case — not that genuine breach risk is absent.

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(simulated_preds, bins=30)
plt.axvline(24, linestyle="--", label="24-hour breach threshold")
plt.axvline(sim_summary["mean_pred"], linestyle="-", label="Mean simulated prediction")
plt.title("Simulated Dwell Distribution — Severe-Breach Example")
plt.xlabel("Predicted dwell hours")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.savefig(figures_dir / "phase6_severe_breach_histogram.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summarize multiple sample cases

Next, the same simulation process is applied to several sample rows from the test set. This gives a small comparison table showing:

- actual observed dwell
- average simulated predicted dwell
- lower and upper uncertainty bands
- estimated breach probability

This is a more planning-oriented summary than a single point forecast alone.

In [20]:
simulation_results = []

for idx, row in sample_cases.iterrows():
    preds, summary = simulate_dwell_distribution(
        model=reg_model,
        row=row,
        feature_cols=feature_cols,
        n_simulations=1000,
        breach_threshold=24.0
    )

    simulation_results.append({
        "date": row["date"],
        "terminal_id": row["terminal_id"],
        "actual_dwell": row["target_dwell_hours"],
        "mean_pred": summary["mean_pred"],
        "p10": summary["p10"],
        "p90": summary["p90"],
        "breach_probability": summary["breach_probability"]
    })

simulation_df = pd.DataFrame(simulation_results)
simulation_df

,date,terminal_id,actual_dwell,mean_pred,p10,p90,breach_probability
0,2024-12-10,T01,15.1,21.013135,20.433984,21.454629,0.000
1,2024-07-22,T08,16.8,15.559022,15.034028,16.219998,0.000
2,2024-08-04,T04,24.1,24.002108,23.298598,24.616667,0.479


## 9. Define stress scenarios

Baseline uncertainty is useful, but operations planning often needs explicit stress tests. This section defines simple scenario multipliers representing operational deterioration or additional pressure.

These are transparent scenario assumptions, not precise real-world engineering rules. Their purpose is to test how the forecasting system responds to higher inbound load or reduced resource availability.

In [21]:
stress_scenarios = {
    "baseline": {
        "inbound_train_count": 1.00,
        "inbound_car_count": 1.00,
        "cars_on_hand": 1.00,
        "yard_occupancy_pct": 1.00,
        "crew_starts_available": 1.00,
        "locomotive_availability_pct": 1.00,
    },
    "volume_stress": {
        "inbound_train_count": 1.10,
        "inbound_car_count": 1.10,
        "cars_on_hand": 1.08,
        "yard_occupancy_pct": 1.05,
        "crew_starts_available": 1.00,
        "locomotive_availability_pct": 1.00,
    },
    "crew_stress": {
        "inbound_train_count": 1.00,
        "inbound_car_count": 1.00,
        "cars_on_hand": 1.00,
        "yard_occupancy_pct": 1.00,
        "crew_starts_available": 0.90,
        "locomotive_availability_pct": 1.00,
    },
    "combined_stress": {
        "inbound_train_count": 1.10,
        "inbound_car_count": 1.10,
        "cars_on_hand": 1.08,
        "yard_occupancy_pct": 1.05,
        "crew_starts_available": 0.90,
        "locomotive_availability_pct": 0.95,
    },
}

stress_scenarios

{'baseline': {'inbound_train_count': 1.0,
  'inbound_car_count': 1.0,
  'cars_on_hand': 1.0,
  'yard_occupancy_pct': 1.0,
  'crew_starts_available': 1.0,
  'locomotive_availability_pct': 1.0},
 'volume_stress': {'inbound_train_count': 1.1,
  'inbound_car_count': 1.1,
  'cars_on_hand': 1.08,
  'yard_occupancy_pct': 1.05,
  'crew_starts_available': 1.0,
  'locomotive_availability_pct': 1.0},
 'crew_stress': {'inbound_train_count': 1.0,
  'inbound_car_count': 1.0,
  'cars_on_hand': 1.0,
  'yard_occupancy_pct': 1.0,
  'crew_starts_available': 0.9,
  'locomotive_availability_pct': 1.0},
 'combined_stress': {'inbound_train_count': 1.1,
  'inbound_car_count': 1.1,
  'cars_on_hand': 1.08,
  'yard_occupancy_pct': 1.05,
  'crew_starts_available': 0.9,
  'locomotive_availability_pct': 0.95}}

## 10. Build a scenario-adjusted simulation function

This version of the simulation function first applies a named scenario to the operational inputs and then adds random perturbation around that stressed state.

This allows us to compare:

- normal operating uncertainty
- stressed operating uncertainty
- changes in breach probability across scenarios

In [ ]:
def simulate_scenario_distribution(
    model,
    row,
    feature_cols,
    scenario_multipliers,
    n_simulations=1000,
    breach_threshold=24.0
):
    base = row[feature_cols]
    n = n_simulations
    m = scenario_multipliers

    sim_df = pd.DataFrame({
        "terminal_id":                 [base["terminal_id"]] * n,
        "inbound_train_count":         np.maximum(0, np.round(
                                           base["inbound_train_count"] * m["inbound_train_count"]
                                           * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "inbound_car_count":           np.maximum(0, np.round(
                                           base["inbound_car_count"] * m["inbound_car_count"]
                                           * np.random.normal(1.00, 0.10, n)
                                       )).astype(int),
        "cars_on_hand":                np.maximum(0, np.round(
                                           base["cars_on_hand"] * m["cars_on_hand"]
                                           * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "yard_occupancy_pct":          np.clip(
                                           base["yard_occupancy_pct"] * m["yard_occupancy_pct"]
                                           * np.random.normal(1.00, 0.05, n),
                                           0, 100
                                       ),
        "crew_starts_available":       np.maximum(0, np.round(
                                           base["crew_starts_available"] * m["crew_starts_available"]
                                           * np.random.normal(1.00, 0.08, n)
                                       )).astype(int),
        "locomotive_availability_pct": np.clip(
                                           base["locomotive_availability_pct"] * m["locomotive_availability_pct"]
                                           * np.random.normal(1.00, 0.04, n),
                                           0, 100
                                       ),
        "is_weekend":                  np.full(n, int(base["is_weekend"])),
        "month":                       np.full(n, int(base["month"])),
    })[feature_cols]

    sim_df["terminal_id"] = sim_df["terminal_id"].astype("category")

    simulated_preds = model.predict(sim_df)

    summary = {
        "mean_pred":          float(simulated_preds.mean()),
        "median_pred":        float(np.median(simulated_preds)),
        "p10":                float(np.percentile(simulated_preds, 10)),
        "p90":                float(np.percentile(simulated_preds, 90)),
        "min_pred":           float(simulated_preds.min()),
        "max_pred":           float(simulated_preds.max()),
        "breach_probability": float((simulated_preds >= breach_threshold).mean()),
    }

    return simulated_preds, summary

## 11. Compare scenarios for the severe-breach example

This comparison applies the four defined scenarios to the severe-breach row. The goal is to observe whether stressed conditions shift the simulated distribution or change the breach probability — or whether the regression model's compression prevents meaningful variation across scenarios.

In [23]:
scenario_results = []

for scenario_name, multipliers in stress_scenarios.items():
    preds, summary = simulate_scenario_distribution(
        model=reg_model,
        row=example_row,
        feature_cols=feature_cols,
        scenario_multipliers=multipliers,
        n_simulations=1000,
        breach_threshold=24.0
    )

    scenario_results.append({
        "scenario": scenario_name,
        "mean_pred": summary["mean_pred"],
        "p10": summary["p10"],
        "p90": summary["p90"],
        "breach_probability": summary["breach_probability"]
    })

scenario_df = pd.DataFrame(scenario_results)
scenario_df

,scenario,mean_pred,p10,p90,breach_probability
0,baseline,20.846890,20.135673,21.440687,0.0
1,volume_stress,20.830366,20.131740,21.439866,0.0
2,crew_stress,20.820855,20.152394,21.445777,0.0
3,combined_stress,21.239236,20.808219,21.558222,0.0


## 12. Plot severe-breach scenario breach probability

This bar chart shows the estimated 24-hour breach probability under each scenario for the severe-breach example. If the regression model compresses the prediction well below the threshold, all scenarios will show near-zero breach probability regardless of the stress applied.

In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(scenario_df["scenario"], scenario_df["breach_probability"])
plt.title("Estimated 24-Hour Breach Probability by Scenario — Severe-Breach Example")
plt.xlabel("Scenario")
plt.ylabel("Breach probability")
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig(figures_dir / "phase6_severe_breach_scenario_probs.png", dpi=150, bbox_inches="tight")
plt.show()

### Important interpretation note

For this example row, the simulation-based breach probability remains near zero across all tested scenarios even though the observed dwell was substantially above 24 hours. This is not a plotting error. Instead, it reflects a known limitation of the tuned regression model: it tends to compress extreme breach cases toward the center of the prediction range. Because the simulation layer is built on top of the regression model, it inherits that limitation.

This result is consistent with the broader project findings. The regression model remains useful for planning-oriented dwell forecasting, but it is not the strongest tool for rare-event breach detection. That is why Phase 5 introduced a dedicated classifier, which produced much stronger breach recall than regression-derived warning flags.

## 13. Prediction-near-threshold planning case

The severe-breach example illustrated a key limitation: when the regression model compresses an extreme realized breach toward the center of its prediction range, the simulation layer inherits that compression and produces near-zero breach probability regardless of the scenario applied.

To complement that finding, this section selects a row based on the model's own predicted dwell value rather than observed dwell alone. The filter looks for rows where the regression model already predicts dwell close to the 24-hour threshold. This places the candidate near the model's operational decision boundary, where simulation-based uncertainty is more likely to produce useful breach probability estimates.

The code below first attaches model predictions to the test set, then filters for rows with predicted dwell in the 22.5–24.5 hour range and selects the one with the highest predicted value within that window.

In [29]:
test_df = test_df.copy()
test_df["predicted_dwell"] = reg_model.predict(X_test)

test_df[[
    "date",
    "terminal_id",
    "target_dwell_hours",
    "predicted_dwell"
]].sort_values("predicted_dwell", ascending=False).head(10)

,date,terminal_id,target_dwell_hours,predicted_dwell
8298,2024-11-03,T03,15.3,25.173199
7923,2024-09-17,T04,29.0,25.114365
7619,2024-08-10,T04,33.7,25.064703
7683,2024-08-18,T04,23.0,24.977219
8507,2024-11-29,T04,25.3,24.851447
7963,2024-09-22,T04,29.8,24.783760
8691,2024-12-22,T04,28.6,24.764627
7987,2024-09-25,T04,27.2,24.753591
8275,2024-10-31,T04,24.0,24.742989
7675,2024-08-17,T04,31.8,24.688459


In [30]:
predicted_risk_cases = test_df[
    (test_df["predicted_dwell"] >= 22.5) &
    (test_df["predicted_dwell"] <= 24.5)
].copy()

print("Prediction-near-threshold candidate rows:", len(predicted_risk_cases))

predicted_case = predicted_risk_cases.sort_values("predicted_dwell", ascending=False).iloc[0].copy()

predicted_case[[
    "date",
    "terminal_id",
    "target_dwell_hours",
    "predicted_dwell",
    "inbound_train_count",
    "inbound_car_count",
    "cars_on_hand",
    "yard_occupancy_pct",
    "crew_starts_available",
    "locomotive_availability_pct"
]]

Prediction-near-threshold candidate rows: 197


date                           2024-09-14 00:00:00
terminal_id                                    T04
target_dwell_hours                            25.2
predicted_dwell                          24.467144
inbound_train_count                              9
inbound_car_count                              194
cars_on_hand                                   855
yard_occupancy_pct                            90.0
crew_starts_available                           12
locomotive_availability_pct                   91.1
Name: 7899, dtype: object

## 14. Compare scenarios for the prediction-near-threshold case

The same four stress scenarios are now applied to the selected prediction-near-threshold row. Because this row sits near the model's 24-hour decision boundary, the simulation should produce meaningful breach probability estimates that differ across operating conditions.

In [31]:
predicted_case_scenario_results = []

for scenario_name, multipliers in stress_scenarios.items():
    preds, summary = simulate_scenario_distribution(
        model=reg_model,
        row=predicted_case,
        feature_cols=feature_cols,
        scenario_multipliers=multipliers,
        n_simulations=1000,
        breach_threshold=24.0
    )

    predicted_case_scenario_results.append({
        "scenario": scenario_name,
        "mean_pred": summary["mean_pred"],
        "p10": summary["p10"],
        "p90": summary["p90"],
        "breach_probability": summary["breach_probability"]
    })

predicted_case_scenario_df = pd.DataFrame(predicted_case_scenario_results)
predicted_case_scenario_df

,scenario,mean_pred,p10,p90,breach_probability
0,baseline,24.342653,23.776557,24.842347,0.813
1,volume_stress,24.261375,23.777400,24.738796,0.750
2,crew_stress,24.275922,23.679373,24.839496,0.760
3,combined_stress,24.227544,23.811174,24.670145,0.716


## 15. Plot prediction-near-threshold breach probability

This bar chart shows the estimated 24-hour breach probability under each scenario for the prediction-near-threshold case.

In [ ]:
plt.figure(figsize=(9, 5))
bars = plt.bar(
    predicted_case_scenario_df["scenario"],
    predicted_case_scenario_df["breach_probability"]
)

plt.title("Estimated 24-Hour Breach Probability by Scenario\n(Prediction-Near-Threshold Case)")
plt.xlabel("Scenario")
plt.ylabel("Breach probability")
plt.ylim(0, 1)

for bar, value in zip(bars, predicted_case_scenario_df["breach_probability"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.savefig(figures_dir / "phase6_threshold_case_scenario_probs.png", dpi=150, bbox_inches="tight")
plt.show()

### Interpretation of the prediction-near-threshold case

This example shows where simulation adds practical planning value. Because the selected row sits near the regression model's 24-hour decision boundary, the simulation produces a substantial estimated breach probability across all tested scenarios. This gives a more decision-relevant picture than a single point forecast alone.

However, the scenario ordering is not perfectly monotonic: the stressed cases do not always produce higher breach probability than baseline. This is not necessarily a coding error. More likely, it reflects a combination of model nonlinearity, local feature interactions, synthetic-data simplifications, and the use of simple scenario multipliers that do not guarantee directional deterioration in every part of the feature space.

This result suggests that simulation can be useful for uncertainty-aware planning, but scenario design should be refined carefully if the goal is to support operational stress testing with stronger directional realism.

## 16. Population-level breach probability distribution

The case studies above examined two specific rows in detail. This section applies the baseline simulation more broadly across a planning-relevant subset of the test set to assess whether the simulation layer produces useful signal at scale.

The subset is defined as rows where the regression model predicts dwell of 20 hours or more — roughly the upper quartile of model predictions, where breach risk is plausible. If this subset exceeds 300 rows, a capped random sample is used and noted explicitly.

In [ ]:
AGGREGATE_CAP = 300

planning_subset = test_df[test_df["predicted_dwell"] >= 20].copy()
print(f"Rows with predicted_dwell >= 20: {len(planning_subset)}")

if len(planning_subset) > AGGREGATE_CAP:
    planning_subset = planning_subset.sample(AGGREGATE_CAP, random_state=42)
    print(f"Capped to {AGGREGATE_CAP} rows (random_state=42) for reasonable runtime.")

aggregate_breach_probs = []
for idx, row in planning_subset.iterrows():
    _, summary = simulate_dwell_distribution(
        model=reg_model,
        row=row,
        feature_cols=feature_cols,
        n_simulations=1000,
        breach_threshold=24.0
    )
    aggregate_breach_probs.append(summary["breach_probability"])

aggregate_breach_probs = np.array(aggregate_breach_probs)

aggregate_summary = pd.Series({
    "n_rows":             int(len(aggregate_breach_probs)),
    "mean_breach_prob":   round(float(aggregate_breach_probs.mean()), 4),
    "median_breach_prob": round(float(np.median(aggregate_breach_probs)), 4),
    "p10_breach_prob":    round(float(np.percentile(aggregate_breach_probs, 10)), 4),
    "p90_breach_prob":    round(float(np.percentile(aggregate_breach_probs, 90)), 4),
    "share_above_0.25":   round(float((aggregate_breach_probs >= 0.25).mean()), 4),
    "share_above_0.50":   round(float((aggregate_breach_probs >= 0.50).mean()), 4),
    "share_above_0.75":   round(float((aggregate_breach_probs >= 0.75).mean()), 4),
})

aggregate_summary

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(aggregate_breach_probs, bins=20, edgecolor="white")
plt.axvline(0.25, linestyle="--", color="orange", linewidth=1.5, label="0.25")
plt.axvline(0.50, linestyle="--", color="red", linewidth=1.5, label="0.50")
plt.axvline(0.75, linestyle="--", color="darkred", linewidth=1.5, label="0.75")
plt.title(
    f"Distribution of Simulated Breach Probability\n"
    f"(Test rows with predicted dwell ≥ 20 hours, n={len(aggregate_breach_probs)})"
)
plt.xlabel("Estimated breach probability")
plt.ylabel("Number of terminal-days")
plt.legend(title="Probability threshold")
plt.tight_layout()
plt.savefig(
    figures_dir / "phase6_population_breach_probability_distribution.png",
    dpi=150, bbox_inches="tight"
)
plt.show()

### Interpretation of the population-level result

The distribution above shows how simulated breach probabilities are spread across the high-predicted-dwell planning subset. A bimodal or polarized pattern — where many terminal-days cluster at very low or very high breach probability with fewer in the middle — is consistent with how LightGBM predictions work: the model tends to produce stable predictions within regions of feature space, so rows either sit comfortably below the threshold or are consistently pushed above it.

Key implications:

- Terminal-days with moderate simulated breach probability (roughly 0.25–0.75) are where uncertainty is most decision-relevant and where the simulation adds the clearest planning value.
- Rows with persistently high breach probability across simulations are strong candidates for proactive operational attention.
- This aggregate view complements the dedicated breach classifier from Phase 5, which was specifically designed for high recall of rare extreme-breach events. Regression-based simulation and a threshold classifier serve different planning roles and should be used together.

## 17. Conclusion and next steps

Phase 6 extended the project from point forecasting into uncertainty-aware planning by adding a simulation layer on top of the tuned LightGBM regression model.

The case studies and population-level analysis illustrated complementary findings:

- **Severe-breach example**: when a realized dwell is far above the threshold, the regression model compresses its prediction toward the center of the range. The simulation inherits that compression, producing near-zero breach probability even under stress. This is consistent with the Phase 5 finding that a dedicated classifier is needed for strong breach recall — regression alone underserves rare-event detection.

- **Prediction-near-threshold example**: when a row sits near the model's 24-hour decision boundary, simulation produces meaningful breach probability estimates and allows comparison across operating scenarios. This is where uncertainty-aware planning adds the most practical value.

- **Population-level view**: applying the simulation layer across high-predicted-dwell terminal-days produces a breach-probability distribution that can help triage which forecasted days warrant closer planning attention.

This simulation layer is a bridge toward richer uncertainty methods, not a final operational simulator. Possible next steps include calibrating scenario multipliers against historical operational records for more directionally realistic stress design, and exploring conformal prediction or quantile regression as alternatives to Monte Carlo simulation.

Translating simulated breach probabilities into cost or operational impact estimates is out of scope for this notebook and is deferred to future work.